# Agent 2 — Step A: Merge & Clean Price Data

**Goal:** Read all 126 Agmarknet CSV files from `price_data/{jowar,bajra,ragi}/`, parse title lines to extract state/month/year, clean rows, and save `master_prices.csv`.

**Before running:** Upload your `price_data/` folder to Google Drive at:
```
MyDrive/MilletSaarthi/price_data/
```
with `jowar/`, `bajra/`, `ragi/` subfolders inside.

## Cell 1 — Mount Drive & verify file counts

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/MilletSaarthi/price_data"
print("Jowar files:", len(os.listdir(f"{BASE}/jowar")))
print("Bajra files:", len(os.listdir(f"{BASE}/bajra")))
print("Ragi  files:", len(os.listdir(f"{BASE}/ragi")))

## Cell 2 — Parse, clean, merge → master_prices.csv

In [ ]:
import pandas as pd
import re
from glob import glob

MONTH_MAP = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}

def parse_file(filepath, millet):
    """Parse one Agmarknet monthly CSV. Extract state/month/year from title line,
    then pull district + modal price rows."""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    if len(lines) < 3:
        return None

    # Title (line 0): '...in <State> - <Month>, <Year>'
    title = lines[0]
    m = re.search(r"in\s+(.+?)\s*-\s*(\w+),\s*(\d{4})", title)
    if not m:
        return None
    state = m.group(1).strip()
    month = MONTH_MAP.get(m.group(2))
    year = int(m.group(3))
    if month is None:
        return None

    rows = []
    for line in lines[2:]:
        line = line.strip()
        if not line or line.lower().startswith(("note", "weighted", "change")):
            break
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 4:
            continue
        district = parts[0]
        if district.lower() in ("district", "average", ""):
            continue
        try:
            price = float(parts[1]) if parts[1] not in ("-", "") else None
        except ValueError:
            price = None
        if price is None:
            continue
        rows.append({
            "millet": millet,
            "state": state,
            "district": district,
            "year": year,
            "month": month,
            "modal_price": price
        })
    return rows

all_rows = []
for millet in ["jowar", "bajra", "ragi"]:
    files = glob(f"{BASE}/{millet}/*.csv")
    for fp in files:
        result = parse_file(fp, millet)
        if result:
            all_rows.extend(result)

df = pd.DataFrame(all_rows)
print("Total rows:", len(df))
print("\nRows per millet:")
print(df["millet"].value_counts())
print("\nRows per state:")
print(df["state"].value_counts())
print("\nYear range:", df["year"].min(), "to", df["year"].max())
print("\nSample rows:")
print(df.head(10))

df.to_csv("/content/drive/MyDrive/MilletSaarthi/master_prices.csv", index=False)
print("\n✅ Saved master_prices.csv")